# Arc balanced-rest pseudobulk audit

This notebook recreates the KaroSpace balanced-rest calculation for `Arc` in the requested annotation categories. KaroSpace fits one shared `~ replicate + annotation` DESeq2 model, then tests each category against the equally weighted mean of every other retained annotation category. This is not a pooled all-other-cells rest comparison.


In [ ]:
# Parameters: edit these for your dataset
H5AD_PATH = "/Users/bastien.herve/Downloads/hipp_celltyped_v3_noLowQ.h5ad"

# GROUPBY is the annotation/color column used for DE Genes, e.g. cell type annotation.
GROUPBY = "major_celltype"

# REPLICATE must match the KaroSpace export groupby/section column, often sample_id.
REPLICATE = "mouse_id"

GENE = "Sox10"
SOURCE_LABELS = ["COP", "Oligodendrocytes"]

# Same defaults as KaroSpace export unless you changed them.
COUNTS_LAYER = "counts"  # set to None to force adata.X
MIN_CELLS = 20
MIN_REPLICATES = 2
MIN_PCT_EXPRESSED = 0.2
P_ADJUST_METHOD = "fdr_bh"
PADJ_CUTOFF = 0.05
LOG2FC_CUTOFF = 0.5
FIT_TYPE = "mean"
DIAGNOSTICS = "pairwise"  # "none" or "pairwise"

In [18]:
from pathlib import Path
import difflib

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

from karospace.pseudobulk import (
    _as_count_matrix,
    _to_dense_counts,
    compute_pseudobulk_group_de,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


def tiny_pseudocount():
    return float(np.nextafter(0.0, 1.0))


def log2_with_tiny_pseudocount(numerator, denominator):
    tiny = tiny_pseudocount()
    a = max(0.0, float(numerator) if np.isfinite(numerator) else 0.0) + tiny
    b = max(0.0, float(denominator) if np.isfinite(denominator) else 0.0) + tiny
    return float(np.log2(a) - np.log2(b))


def find_category(categories, wanted):
    """Resolve labels robustly while still making mismatches explicit."""
    wanted = str(wanted)
    categories = [str(c) for c in categories]
    if wanted in categories:
        return wanted
    lower = {c.lower(): c for c in categories}
    if wanted.lower() in lower:
        return lower[wanted.lower()]
    contains = [c for c in categories if wanted.lower() in c.lower() or c.lower() in wanted.lower()]
    if len(contains) == 1:
        return contains[0]
    close = difflib.get_close_matches(wanted, categories, n=8, cutoff=0.35)
    raise KeyError(
        f"Could not resolve category {wanted!r}. Close matches: {close}. "
        "Set SOURCE_LABELS to the exact category names from adata.obs[GROUPBY].cat.categories."
    )


def find_gene(var_names, wanted):
    genes = [str(g) for g in var_names]
    if wanted in genes:
        return wanted
    lower = {g.lower(): g for g in genes}
    if wanted.lower() in lower:
        return lower[wanted.lower()]
    close = difflib.get_close_matches(wanted, genes, n=8, cutoff=0.5)
    raise KeyError(f"Gene {wanted!r} not found. Close matches: {close}")


def one_gene_result(result, gene):
    if not result:
        return {"available": False, "reason": "missing_result"}
    if result.get("available") is False:
        return {
            "available": False,
            "reason": result.get("reason"),
            "details": result.get("details"),
            "n_source": result.get("n_source"),
            "n_reference": result.get("n_reference"),
            "min_cells_required": result.get("min_cells_required"),
            "min_replicates_required": result.get("min_replicates_required"),
        }
    genes = [str(g) for g in result.get("genes", [])]
    if gene not in genes:
        return {
            "available": True,
            "gene_present": False,
            "n_result_genes": len(genes),
            "method": result.get("method"),
            "min_pct_expressed": result.get("min_pct_expressed"),
        }
    i = genes.index(gene)
    log2fc = result.get("log2foldchanges", result.get("logfoldchanges", []))[i]
    padj = result.get("pvals_adj", [None] * len(genes))[i]
    return {
        "available": True,
        "gene_present": True,
        "method": result.get("method"),
        "counts_layer": result.get("counts_layer"),
        "n_source": result.get("n_source"),
        "n_reference": result.get("n_reference"),
        "n_replicates": result.get("n_replicates"),
        "base_mean": result.get("base_mean", [None] * len(genes))[i],
        "log2FC": log2fc,
        "stat": result.get("scores", [None] * len(genes))[i],
        "pvalue": result.get("pvals", [None] * len(genes))[i],
        "padj": padj,
        "pct_source": result.get("pct_source", [None] * len(genes))[i],
        "pct_rest": result.get("pct_reference", [None] * len(genes))[i],
        "padj_cutoff": result.get("padj_cutoff"),
        "log2fc_cutoff": result.get("log2fc_cutoff"),
        "passes_DE_star": (padj is not None and log2fc is not None and padj <= result.get("padj_cutoff", 0.05) and log2fc >= result.get("log2fc_cutoff", 0.5)),
    }

In [19]:
adata = ad.read_h5ad(H5AD_PATH)
print(adata)

missing_obs = [c for c in [GROUPBY, REPLICATE] if c not in adata.obs.columns]
if missing_obs:
    raise KeyError(f"Missing obs columns: {missing_obs}")

group_col = adata.obs[GROUPBY]
if not isinstance(group_col.dtype, pd.CategoricalDtype):
    group_col = group_col.astype("category")
categories = [str(c) for c in group_col.cat.categories]
gene = find_gene(adata.var_names, GENE)
source_labels = [find_category(categories, label) for label in SOURCE_LABELS]

print(f"Gene: {gene}")
print(f"GROUPBY={GROUPBY!r}: {len(categories)} categories")
print(source_labels)
pd.Series(categories, name="categories").to_frame().head(50)

AnnData object with n_obs × n_vars = 1713896 × 347
    obs: 'cell_id_old', 'x_centroid', 'y_centroid', 'Xenium_run', 'cell_id_new', 'inside_polygon', 'strain', 'mouse_id', 'region', 'class_label', 'class_name', 'class_bootstrapping_probability', 'subclass_label', 'subclass_name', 'subclass_bootstrapping_probability', 'supertype_label', 'supertype_name', 'supertype_bootstrapping_probability', 'cluster_label', 'cluster_name', 'cluster_alias', 'cluster_bootstrapping_probability', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'n_genes', 'doublet_score', 'predicted_doublet', 'is_it_doublet', 'Batch', 'Slide', 'class_name_high_prob', 'subclass_name_high_prob', 'leiden_1', 'major_celltype', 'final_celltype_annotation', 'final_celltype_annotation_v2', 'is_low_quality', 'final_celltype_annotation_v3'
    var: 'n_cells_by_cou

,categories
0,Astrocytes
1,COP
2,Choroid plexus
3,Endothelial cells
4,Ependymal cells
5,Excitatory neurons
6,Inhibitory neurons
7,Microglia / perivascular macrophages
8,OPC
9,Oligodendrocytes


In [20]:
# Run the same KaroSpace shared-fit balanced-rest pseudobulk DE function used by export_to_html.
de = compute_pseudobulk_group_de(
    adata,
    GROUPBY,
    replicate=REPLICATE,
    counts_layer=COUNTS_LAYER,
    min_cells=MIN_CELLS,
    min_replicates=MIN_REPLICATES,
    min_pct_expressed=MIN_PCT_EXPRESSED,
    p_adjust_method=P_ADJUST_METHOD,
    padj_cutoff=PADJ_CUTOFF,
    log2fc_cutoff=LOG2FC_CUTOFF,
    fit_type=FIT_TYPE,
    diagnostics=DIAGNOSTICS,
)

rows = []
for source in source_labels:
    result = ((de or {}).get(source) or {}).get("__rest__")
    rows.append({"source": source, "reference": "balanced rest", **one_gene_result(result, gene)})
summary = pd.DataFrame(rows)
summary


    - 12 categories -> 132 directed comparisons plus 12 category-vs-rest contrasts (66 pairwise DESeq2 fits)
      - Astrocytes vs COP: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Choroid plexus: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Endothelial cells: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Ependymal cells: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Excitatory neurons: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Inhibitory neurons: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs Microglia / perivascular macrophages: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)
      - Astrocytes vs OPC: fitting DESeq2 (28 paired replicates, 56 pseudobulk samples, 347 genes)

,source,reference,available,gene_present,method,counts_layer,n_source,n_reference,n_replicates,base_mean,log2FC,stat,pvalue,padj,pct_source,pct_rest,padj_cutoff,log2fc_cutoff,passes_DE_star
0,COP,__rest__,True,True,pseudobulk-deseq2,counts,8190,1705706,28,9054.055210,2.598985,93.241430,0.0,4.940656e-324,0.771551,0.163532,0.05,0.5,True
1,Oligodendrocytes,__rest__,True,True,pseudobulk-deseq2,counts,283292,1430604,28,54709.789108,5.094992,168.064905,0.0,4.940656e-324,0.807764,0.039441,0.05,0.5,True


## Pairwise diagnostics for Arc

With `DIAGNOSTICS = "pairwise"`, KaroSpace stores one PCA/distance payload for each unordered selected pair. It contains only the two annotations and is reused for either contrast direction. `"none"` omits these diagnostics from the exported HTML. Pairwise diagnostics apply only to category-versus-category contrasts; balanced-rest contrasts deliberately omit them because their reference comprises multiple annotations.


In [13]:
expression_matrix, counts_layer_used, warning = _as_count_matrix(adata, COUNTS_LAYER)
if warning:
    print("Warning:", warning)

group_values = group_col.astype(str)
rep_values = adata.obs[REPLICATE].astype(str)
valid = rep_values.notna().to_numpy() & group_values.notna().to_numpy()
valid &= np.asarray(group_col.cat.codes.to_numpy() >= 0, dtype=bool)
valid_indices = np.flatnonzero(valid)

rep_valid = rep_values.iloc[valid_indices].astype(str).to_numpy()
group_valid = group_values.iloc[valid_indices].astype(str).to_numpy()

sample_keys = []
sample_index = {}
rows = np.empty(valid_indices.size, dtype=np.int64)
for i, (rep, grp) in enumerate(zip(rep_valid, group_valid)):
    key = (str(rep), str(grp))
    idx = sample_index.get(key)
    if idx is None:
        idx = len(sample_keys)
        sample_index[key] = idx
        sample_keys.append(key)
    rows[i] = idx

incidence = sp.csr_matrix(
    (np.ones(valid_indices.size, dtype=np.float64), (rows, valid_indices)),
    shape=(len(sample_keys), adata.n_obs),
)
aggregate = _to_dense_counts(incidence @ expression_matrix)
cell_counts = np.bincount(rows, minlength=len(sample_keys)).astype(int)
pb_meta = pd.DataFrame(
    {
        "replicate": [k[0] for k in sample_keys],
        "category": [k[1] for k in sample_keys],
        "n_cells": cell_counts,
    },
    index=[f"pb_{i}" for i in range(len(sample_keys))],
)

gene_idx = [str(g) for g in adata.var_names].index(gene)
pb_meta["gene_counts"] = aggregate[:, gene_idx]
pb_meta["gene_mean_per_cell"] = pb_meta["gene_counts"] / pb_meta["n_cells"].replace(0, np.nan)

audit_rows = []
for source in source_labels:
    source_pb = pb_meta[(pb_meta["category"] == source) & (pb_meta["n_cells"] >= MIN_CELLS)]
    for rep in sorted(source_pb["replicate"].astype(str).unique()):
        src = pb_meta[(pb_meta["replicate"].astype(str) == rep) & (pb_meta["category"] == source) & (pb_meta["n_cells"] >= MIN_CELLS)]
        rest = pb_meta[(pb_meta["replicate"].astype(str) == rep) & (pb_meta["category"] != source)]
        rest_cells = int(rest["n_cells"].sum())
        if src.empty or rest.empty or rest_cells < MIN_CELLS:
            continue
        audit_rows.append(
            {
                "source": source,
                "replicate": rep,
                "source_cells": int(src["n_cells"].iloc[0]),
                "source_arc_counts": int(src["gene_counts"].iloc[0]),
                "source_arc_mean": float(src["gene_mean_per_cell"].iloc[0]),
                "rest_cells": rest_cells,
                "rest_arc_counts": int(rest["gene_counts"].sum()),
                "rest_arc_mean": float(rest["gene_counts"].sum() / rest_cells),
                "rough_log2fc_tiny_pseudocount": log2_with_tiny_pseudocount(src["gene_counts"].iloc[0] / src["n_cells"].iloc[0], rest["gene_counts"].sum() / rest_cells),
            }
        )

replicate_audit = pd.DataFrame(audit_rows)
replicate_audit

,source,replicate,source_cells,source_arc_counts,source_arc_mean,rest_cells,rest_arc_counts,rest_arc_mean,rough_log2fc_tiny_pseudocount
0,COP,261982,273,701,2.567766,57008,25054,0.439482,2.546637
1,COP,264010,257,525,2.042802,58554,20164,0.344366,2.568535
2,COP,264011,266,655,2.462406,61428,20688,0.336785,2.870171
3,COP,311430,244,720,2.950820,58839,24308,0.413127,2.836457
4,COP,350670,244,761,3.118852,62515,27675,0.442694,2.816634
5,COP,437942,293,630,2.150171,61443,25268,0.411243,2.386388
6,COP,437943,282,753,2.670213,60251,23475,0.389620,2.776815
7,COP,437944,346,772,2.231214,67048,26154,0.390079,2.515991
8,COP,437968,225,540,2.400000,59742,21373,0.357755,2.745991
9,COP,439247,341,843,2.472141,74288,34103,0.459065,2.428991


## Balanced-rest calculation

For each source category, this notebook shows cell-level descriptive values and the KaroSpace result from the shared model. The rough pooled rest mean is only descriptive: it is not the DESeq2 balanced-rest reference.


In [14]:
from IPython.display import display

def gene_values_from_matrix(matrix, mask, gene_idx):
    if sp.issparse(matrix):
        return np.asarray(matrix[mask, gene_idx].toarray()).ravel()
    return np.asarray(matrix[mask, gene_idx]).ravel()

def breakdown_balanced_rest(source):
    source = str(source)
    result = ((de or {}).get(source) or {}).get("__rest__")
    source_mask = (group_values.to_numpy() == source) & valid
    other_mask = (group_values.to_numpy() != source) & valid
    source_values = gene_values_from_matrix(expression_matrix, source_mask, gene_idx)
    other_values = gene_values_from_matrix(expression_matrix, other_mask, gene_idx)

    descriptive = pd.DataFrame([{
        "source": source,
        "source_cells": int(source_mask.sum()),
        "all_other_cells": int(other_mask.sum()),
        "source_mean_per_cell": float(source_values.mean()) if source_values.size else np.nan,
        "pooled_other_mean_per_cell": float(other_values.mean()) if other_values.size else np.nan,
        "note": "The pooled-other mean is descriptive only; DESeq2 uses an equal-category-weight balanced-rest contrast.",
    }])
    display(descriptive)

    arc_result = one_gene_result(result, gene)
    display(pd.DataFrame([{"source": source, "reference": "balanced rest", **arc_result}]))

    diagnostics = None  # Balanced-rest contrasts deliberately have no pairwise PCA/distance payload.
    if diagnostics:
        display(pd.DataFrame({
            "label": diagnostics.get("labels", []),
            "replicate": diagnostics.get("replicates", []),
            "group": diagnostics.get("groups", []),
            "n_cells": diagnostics.get("n_cells", []),
            "library_size": diagnostics.get("library_size", []),
        }))
    return {
        "source": source,
        "source_mean": descriptive.loc[0, "source_mean_per_cell"],
        "rest_mean": descriptive.loc[0, "pooled_other_mean_per_cell"],
        "included_replicates": sorted(set((diagnostics or {}).get("replicates", []))),
        "de_result": arc_result,
    }


In [15]:
breakdowns = {source: breakdown_balanced_rest(source) for source in source_labels}

comparison_rows = []
for source, payload in breakdowns.items():
    de_result = payload["de_result"]
    comparison_rows.append({
        "source": source,
        "descriptive_source_mean": payload["source_mean"],
        "descriptive_pooled_other_mean": payload["rest_mean"],
        "DESeq2_balanced_rest_log2FC": de_result.get("log2FC"),
        "DESeq2_padj": de_result.get("padj"),
        "pct_source": de_result.get("pct_source"),
        "pct_reference": de_result.get("pct_rest"),
        "passes_DE_star": de_result.get("passes_DE_star"),
        "fit_replicates": ", ".join(payload["included_replicates"]),
    })
pd.DataFrame(comparison_rows)


Sox10: COP vs rest


,step,category,cells,Arc_positive_cells,pct_Arc_positive,Arc_counts_sum,Arc_mean_per_cell
0,1. source cells,COP,8190,6319,0.771551,20152.0,2.460562
1,1. rest cells,all non-source categories,1705706,278938,0.163532,692733.0,0.406127


,step,source_mean,rest_mean,log2(source_mean / rest_mean),log2_with_tiny_pseudocount,note
0,2. pooled rough log2FC,2.460562,0.406127,2.598985,2.598985,This is not DESeq2; it is a rough pooled check.


Step 3. Pooled category components that make up source and rest


,is_source,category,cells,Arc_counts,Arc_mean_per_cell
11,True,COP,8190,20152,2.460562
8,False,Oligodendrocytes,283292,621113,2.192483
7,False,OPC,61452,46037,0.749154
6,False,Microglia / perivascular macrophages,70664,3589,0.050790
3,False,Ependymal cells,29216,895,0.030634
9,False,Pericytes / smooth muscle cells,52581,1479,0.028128
0,False,Astrocytes,244494,6274,0.025661
2,False,Endothelial cells,170126,3728,0.021913
5,False,Inhibitory neurons,198530,2925,0.014733
4,False,Excitatory neurons,531760,6248,0.011750


...

Step 5. KaroSpace-formatted DE result for Arc


,source,reference,available,gene_present,n_result_genes,method,min_pct_expressed,rank_in_sorted_result
0,Oligodendrocytes,__rest__,True,False,89,pseudobulk-deseq2,0.1,None


Step 6. DESeq2 sample diagnostics embedded by KaroSpace


,label,replicate,group,n_cells,library_size
0,261982 | Oligodendrocytes,261982,Oligodendrocytes,9121,822830
1,261982 | __rest__,261982,__rest__,48160,4969023
2,264010 | Oligodendrocytes,264010,Oligodendrocytes,9122,791386
3,264010 | __rest__,264010,__rest__,49689,5073431
4,264011 | Oligodendrocytes,264011,Oligodendrocytes,9268,764705
5,264011 | __rest__,264011,__rest__,52426,5907280
6,311430 | Oligodendrocytes,311430,Oligodendrocytes,10002,907283
7,311430 | __rest__,311430,__rest__,49081,5759556
8,350670 | Oligodendrocytes,350670,Oligodendrocytes,9846,984114
9,350670 | __rest__,350670,__rest__,52913,6048331


,source,source_mean,rest_mean,rough_log2fc_tiny_pseudocount,DESeq2_log2FC,DESeq2_padj,pct_source,pct_rest,passes_DE_star,included_replicates
0,COP,2.460562,0.406127,2.598985,2.598985,4.940656e-324,0.771551,0.163532,True,"261982, 264010, 264011, 311430, 350670, 437942..."
1,Oligodendrocytes,2.192483,0.064149,5.094992,NaN,NaN,NaN,NaN,None,"261982, 264010, 264011, 311430, 350670, 437942..."


In [16]:
# Pooled category means for Arc. These correspond to the values used by DATA.cluster_gene_means.
pooled = (
    pb_meta.groupby("category", observed=True)
    .agg(n_cells=("n_cells", "sum"), gene_counts=("gene_counts", "sum"))
    .reset_index()
)
pooled["mean_per_cell"] = pooled["gene_counts"] / pooled["n_cells"].replace(0, np.nan)
pooled["is_target_source"] = pooled["category"].isin(source_labels)
pooled.sort_values(["is_target_source", "mean_per_cell"], ascending=[False, False]).head(30)

,category,n_cells,gene_counts,mean_per_cell,is_target_source
1,COP,8190,20152,2.460562,True
9,Oligodendrocytes,283292,621113,2.192483,True
8,OPC,61452,46037,0.749154,False
7,Microglia / perivascular macrophages,70664,3589,0.050790,False
4,Ependymal cells,29216,895,0.030634,False
10,Pericytes / smooth muscle cells,52581,1479,0.028128,False
0,Astrocytes,244494,6274,0.025661,False
3,Endothelial cells,170126,3728,0.021913,False
6,Inhibitory neurons,198530,2925,0.014733,False
5,Excitatory neurons,531760,6248,0.011750,False


## Interpretation checklist

- A balanced-rest result compares one category with the equally weighted mean of the other retained annotation coefficients from the shared model.
- The displayed pooled all-other-cells mean is descriptive only and should not be interpreted as the tested DESeq2 reference.
- If `available=False`, inspect `reason`, `details`, `n_source`, and `n_reference`.
- If log2FC is high but adjusted p-value is above the cutoff, replicate-level variability or multiple-testing correction can explain the result.
